# WaveCoAtNet Account 2: Ablation Study (10 Conditions)
> Run this in Account 2. Set Runtime > GPU before starting.
> All outputs auto-save to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/WaveCoAtNet_experiments/wave-coAtNet/wave-CoAtNet'

if not os.path.exists(WORK):
    os.makedirs('/content/drive/MyDrive/WaveCoAtNet_experiments', exist_ok=True)
    %cd /content/drive/MyDrive/WaveCoAtNet_experiments
    !git clone https://github.com/Cyrax321/wave-coAtNet.git
else:
    %cd /content/drive/MyDrive/WaveCoAtNet_experiments/wave-coAtNet
    !git pull origin main

%cd {WORK}
!pip install -q -r requirements.txt

# Runs ALL 10 conditions automatically (~6h)
!python evaluation/ablation.py

In [ ]:
import os, csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
    confusion_matrix, classification_report)
from scipy.stats import chi2 as chi2_dist
from IPython.display import display, Image

os.chdir('/content/drive/MyDrive/WaveCoAtNet_experiments/wave-coAtNet/wave-CoAtNet')
os.makedirs('figures', exist_ok=True)

class_names = ['Harlequin ichthyosis', 'Healthy skin', 'Ichthyosis vulgaris',
               'Lamellar ichthyosis', 'Netherton syndrome']
n_classes = len(class_names)

CONDITIONS = [
    ('full',           'WaveCoAtNet (Full)'),
    ('no_dpa',         'w/o DPA'),
    ('no_pgap',        'w/o PGAP+DPA'),
    ('no_wgfdca',      'w/o WG-FDCA'),
    ('no_transformer', 'w/o Transformer'),
    ('no_padts',       'w/o PA-DTS (GAP)'),
    ('no_sctr',        'w/o SCTR'),
    ('fixed_pruning',  'w/ Fixed Pruning'),
    ('no_prototypes',  'w/o Prototypes'),
    ('baseline',       'ConvNeXt-Tiny Baseline'),
]

results = []
for cond, label in CONDITIONS:
    yt_f = f'ablation_{cond}_y_true.npy'
    yp_f = f'ablation_{cond}_y_pred.npy'
    if os.path.exists(yt_f) and os.path.exists(yp_f):
        results.append((cond, label, np.load(yt_f), np.load(yp_f)))
        print(f"  Loaded: {cond}")
    else:
        print(f"  MISSING: {cond}")

print(f"\nLoaded {len(results)}/10 conditions\n")

# Comprehensive ablation table with deltas
print("="*80)
print("  ABLATION RESULTS TABLE")
print("="*80)

full_acc, full_f1 = None, None
print(f"{'Condition':<25s} {'Acc%':>7s} {'dAcc':>7s} {'F1%':>7s} {'dF1':>7s} {'Kappa':>7s}")
print("-"*60)

for cond, label, yt, yp in results:
    acc = accuracy_score(yt, yp) * 100
    f1 = f1_score(yt, yp, average='macro', zero_division=0) * 100
    kappa = cohen_kappa_score(yt, yp)
    if cond == 'full':
        full_acc, full_f1 = acc, f1
        d_acc, d_f1 = '-', '-'
    else:
        d_acc = f"{acc - full_acc:+.2f}"
        d_f1 = f"{f1 - full_f1:+.2f}"
    print(f"{label:<25s} {acc:>6.2f} {d_acc:>7s} {f1:>6.2f} {d_f1:>7s} {kappa:>6.4f}")

# Ablation bar chart
labels_plot = [r[1] for r in results]
accs_plot = [accuracy_score(r[2], r[3])*100 for r in results]
f1s_plot = [f1_score(r[2], r[3], average='macro', zero_division=0)*100 for r in results]

x = np.arange(len(labels_plot))
w = 0.35
fig, ax = plt.subplots(figsize=(14, 7))
acc_colors = ['#2563EB' if 'Full' in l else '#93C5FD' for l in labels_plot]
f1_colors = ['#DC2626' if 'Full' in l else '#FCA5A5' for l in labels_plot]
bars1 = ax.bar(x - w/2, accs_plot, w, color=acc_colors, alpha=0.85, label='Accuracy (%)')
bars2 = ax.bar(x + w/2, f1s_plot, w, color=f1_colors, alpha=0.85, label='Macro F1 (%)')

for bar in list(bars1) + list(bars2):
    ax.annotate(f'{bar.get_height():.1f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=7, fontweight='bold')

if full_acc is not None:
    for i, l in enumerate(labels_plot):
        if 'Full' not in l:
            d = accs_plot[i] - full_acc
            color = '#DC2626' if d < 0 else '#16A34A'
            ax.annotate(f'{d:+.1f}%', xy=(x[i] - w/2, accs_plot[i]),
                        xytext=(0, -15), textcoords='offset points',
                        ha='center', fontsize=7, color=color, fontstyle='italic', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels_plot, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_title('Ablation Study: Contribution of Each Novel Module', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('figures/ablation_bar_detailed.png', dpi=300, bbox_inches='tight')
plt.close()
print("\nSaved: figures/ablation_bar_detailed.png")

# Ablation per-class F1 heatmap
f1_matrix = []
cond_labels = []
for cond, label, yt, yp in results:
    per_class = f1_score(yt, yp, average=None, zero_division=0, labels=range(n_classes)) * 100
    f1_matrix.append(per_class)
    cond_labels.append(label)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(np.array(f1_matrix), annot=True, fmt='.1f', cmap='YlGnBu',
            xticklabels=[c[:15] for c in class_names], yticklabels=cond_labels,
            annot_kws={"size": 9}, linewidths=0.5, linecolor='white',
            vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'F1 Score (%)'})
ax.set_title('Ablation: Per-Class F1 Scores', fontsize=13, fontweight='bold')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('figures/ablation_f1_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: figures/ablation_f1_heatmap.png")

# McNemar: full vs each ablation
print(f"\n{'='*80}")
print("  McNEMAR: Full vs Each Ablation")
print(f"{'='*80}")

full_yt, full_yp = None, None
for cond, label, yt, yp in results:
    if cond == 'full':
        full_yt, full_yp = yt, yp
        break

if full_yt is not None:
    mcnemar_rows = []
    for cond, label, yt, yp in results:
        if cond == 'full': continue
        correct_full = (full_yt == full_yp).astype(int)
        correct_abl = (yt == yp).astype(int)
        b = np.sum((correct_full == 1) & (correct_abl == 0))
        c = np.sum((correct_full == 0) & (correct_abl == 1))
        if b + c == 0:
            chi2_val, pval = 0.0, 1.0
        else:
            chi2_val = (abs(b - c) - 1) ** 2 / (b + c)
            pval = 1 - chi2_dist.cdf(chi2_val, df=1)
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
        print(f"  {label:<25s} chi2={chi2_val:.3f}  p={pval:.4f}  {sig}")
        mcnemar_rows.append({'condition': label, 'chi2': chi2_val, 'p_value': pval, 'sig': sig})

    with open('ablation_mcnemar.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['condition','chi2','p_value','sig'])
        writer.writeheader()
        writer.writerows(mcnemar_rows)
    print("Saved: ablation_mcnemar.csv")

# Ablation CM grid
cm_files = [f'ablation_{c}_cm.png' for c, _ in CONDITIONS if os.path.exists(f'ablation_{c}_cm.png')]
if len(cm_files) >= 4:
    fig, axes = plt.subplots(2, 5, figsize=(30, 12))
    for i, cm_f in enumerate(cm_files[:10]):
        ax = axes[i//5, i%5]
        ax.imshow(plt.imread(cm_f))
        ax.axis('off')
    for i in range(len(cm_files), 10):
        axes[i//5, i%5].axis('off')
    fig.suptitle('Ablation: Confusion Matrices (All Conditions)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/ablation_cm_grid.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: figures/ablation_cm_grid.png")

for f in ['figures/ablation_bar_detailed.png', 'figures/ablation_f1_heatmap.png', 'figures/ablation_cm_grid.png']:
    if os.path.exists(f):
        display(Image(filename=f, width=700))